# CAPM Beta Estimation & Rolling Risk Analysis

**Goal:** Estimate systematic risk (beta) for a stock relative to the market, then examine how that risk exposure changes over time using a rolling regression.

This mirrors a core task for a market/portfolio risk analyst: monitoring whether a position's market sensitivity is stable or drifting.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import build_dataset
from capm_model import run_static_capm, run_rolling_capm

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Pull Data

Change `STOCK` to whichever ticker you want to analyze. Try picking one high-beta name (e.g., a tech stock) and comparing it against a defensive name (e.g., a utility) across two notebook runs.

In [ ]:
STOCK = 'AAPL'
MARKET = '^GSPC'
START = '2015-01-01'
END = '2025-01-01'
FREQ = 'W'

df = build_dataset(STOCK, MARKET, START, END, FREQ)
df.head()

## 2. Static CAPM Regression

Full-period beta estimate. This is the textbook CAPM number — useful as a baseline, but it hides any time variation.

In [ ]:
static_result = run_static_capm(df)

print(f"Alpha: {static_result['alpha']:.5f}")
print(f"Beta:  {static_result['beta']:.3f}")
print(f"R-squared: {static_result['r_squared']:.3f}")
print(f"Beta p-value: {static_result['p_value_beta']:.4f}")
print()
print(static_result['model'].summary())

### Scatter Plot with Regression Line

In [ ]:
fig, ax = plt.subplots()
sns.regplot(x='market_return', y='stock_return', data=df, ax=ax,
            scatter_kws={'alpha': 0.4}, line_kws={'color': 'red'})
ax.set_title(f'{STOCK} vs Market Returns (Beta = {static_result["beta"]:.2f})')
ax.set_xlabel('Market Return')
ax.set_ylabel(f'{STOCK} Return')
plt.tight_layout()
plt.savefig('../outputs/scatter_regression.png', dpi=150)
plt.show()

## 3. Rolling Beta — The Risk-Relevant Part

A single static beta tells you nothing about whether risk exposure has been stable. This is where the project becomes genuinely useful for a risk role: we re-estimate beta on a rolling window to see how it evolves.

In [ ]:
WINDOW = 52  # ~1 year of weekly data

rolling_df = run_rolling_capm(df, window=WINDOW)
rolling_df.tail()

In [ ]:
fig, ax = plt.subplots()
ax.plot(rolling_df.index, rolling_df['beta'], color='steelblue', linewidth=1.5)
ax.axhline(static_result['beta'], color='red', linestyle='--', alpha=0.6, label='Full-period beta')
ax.axhline(1.0, color='gray', linestyle=':', alpha=0.5, label='Market beta = 1.0')
ax.set_title(f'{STOCK}: Rolling {WINDOW}-Week Beta vs. S&P 500')
ax.set_xlabel('Date')
ax.set_ylabel('Beta')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/rolling_beta.png', dpi=150)
plt.show()

## 4. Diagnostics

Quick check on residuals — a real risk analyst would not stop at R² alone.

In [ ]:
residuals = static_result['model'].resid

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df.index, residuals)
axes[0].set_title('Residuals Over Time')
axes[0].axhline(0, color='red', linestyle='--', alpha=0.5)

axes[1].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('../outputs/residual_diagnostics.png', dpi=150)
plt.show()

## 5. Findings & Risk Interpretation

_Fill this in after reviewing your specific ticker's output. Prompts to answer:_

- What was the full-period beta, and was it statistically significant?
- Did rolling beta stay close to the full-period estimate, or did it swing meaningfully?
- Can you tie any beta shifts to a real market event (e.g., COVID crash, 2022 rate hikes)?
- What would you flag to a portfolio manager based on this? (e.g., "This position's market sensitivity nearly doubled during the 2022 drawdown — worth revisiting position sizing.")